# Глава 5. Предварительное обучение на неразмеченных данных

In [1]:
pip install matplotlib numpy tiktoken torch tensorflow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from importlib.metadata import version

pkgs = ["matplotlib", 
        "numpy", 
        "tiktoken", 
        "torch",
        "tensorflow" # Для предварительно обученных моделей OpenAI
       ]
for p in pkgs:
    print(f"{p} Версия: {version(p)}")

matplotlib Версия: 3.10.9
numpy Версия: 2.4.4
tiktoken Версия: 0.12.0
torch Версия: 2.12.0
tensorflow Версия: 2.21.0


- В этой главе мы реализуем цикл обучения и код для базовой оценки модели, чтобы провести предварительное обучение большой языковой модели
- В конце мы также загружаем в нашу модель общедоступные предварительно обученные веса от OpenAI

<img src="https://camo.githubusercontent.com/137f57f6192fbcb6627e6ced1b5274c71924774dec18a4dea29b1c156619ef24/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830355f636f6d707265737365642f30312e77656270" width=800px>

- Ниже перечислены темы, затронутые в этой главе

<img src="https://camo.githubusercontent.com/01ebc99e37dddc617ba6dd20799f945fd6a562bac8b8abe5a4b82eb491ebbfca/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830355f636f6d707265737365642f30322e77656270" width=800px>

&nbsp;
## 5.1 Оценка генеративных текстовых моделей

- В начале этого раздела мы кратко расскажем о том, как инициализировать модель GPT с помощью кода из предыдущей главы
- Затем мы обсудим основные метрики оценки больших языковых моделей
- Наконец, в этом разделе мы применим эти метрики оценки к обучающему и проверочному наборам данных

&nbsp;
### 5.1.1 Использование GPT для генерации текста

- Мы инициализируем модель GPT с помощью кода из предыдущей главы

In [3]:
import torch
from previous_chapters import GPTModel
# Если файл `previous_chapters.py` недоступен локально,
# вы можете импортировать его из пакета PyPI `llms-from-scratch`. 
# Подробнее см.: https://github.com/rasbt/LLMs-from-scratch/tree/main/pkg
# Например,
# from llms_from_scratch.ch04 import GPTModel

GPT_CONFIG_124M = {
    "vocab_size": 50257,   # Размер словаря
    "context_length": 256, # Сокращенная длина контекста (исходное значение: 1024)
    "emb_dim": 768,        # Размерность эмбеддинга
    "n_heads": 12,         # Количество ядер внимания
    "n_layers": 12,        # Количество слоев
    "drop_rate": 0.1,      # Коэффициент дропаута
    "qkv_bias": False      # Смещение в сторону значений ключей запроса
}

torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
model.eval();  # Отключите дропаут во время логического вывода

- Мы используем дропаут 0,1, но в настоящее время довольно часто обучают большие языковые модели без дропаута
- В современных больших языковых моделях также не используются векторы смещения в слоях `nn.Linear` для матриц запросов, ключей и значений (в отличие от более ранних моделей GPT). Это достигается за счет установки параметра `"qkv_bias": False`
- Мы уменьшили длину контекста (`context_length`) всего на 256 токенов, чтобы снизить требования к вычислительным ресурсам для обучения модели, в то время как исходная модель GPT-2 со 124 миллионами параметров использовала 1024 токена
    - Это сделано для того, чтобы мы могли следить за примерами кода и выполнять их на своих портативных компьютерах
    - Позже мы также загрузим модель с `context_length` 1024 из предварительно обученных весов.

- Далее мы используем функцию `generate_text_simple` из предыдущей главы для генерации текста
- Кроме того, мы определяем две вспомогательные функции: `text_to_token_ids` и `token_ids_to_text` — для преобразования токенов в текстовое представление и обратно, которые мы будем использовать на протяжении всей главы

<img src="https://camo.githubusercontent.com/0756c65e7e7878cab7b7673bedd3f441cf42f1f67e89b46d9e7ec931e0fffbc3/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830355f636f6d707265737365642f30332e77656270" width=800px>

In [4]:
import tiktoken
from previous_chapters import generate_text_simple

def text_to_token_ids(text, tokenizer):
    encoded = tokenizer.encode(text, allowed_special={'<|endoftext|>'})
    encoded_tensor = torch.tensor(encoded).unsqueeze(0) # добавить размер пакета
    return encoded_tensor

def token_ids_to_text(token_ids, tokenizer):
    flat = token_ids.squeeze(0) # удалить размер пакета
    return tokenizer.decode(flat.tolist())

start_context = "Every effort moves you"
tokenizer = tiktoken.get_encoding("gpt2")

token_ids = generate_text_simple(
    model=model,
    idx=text_to_token_ids(start_context, tokenizer),
    max_new_tokens=10,
    context_size=GPT_CONFIG_124M["context_length"]
)

print("Выводимый текст:\n", token_ids_to_text(token_ids, tokenizer))

Выводимый текст:
 Every effort moves you rentingetic wasnم refres RexMeCHicular stren


- Как мы видим выше, модель не генерирует качественный текст, потому что она еще не обучена
- Как измерить или зафиксировать в числовом выражении, что такое «качественный текст», чтобы отслеживать этот показатель во время обучения?
- В следующем подразделе мы рассмотрим метрики для расчета показателя потерь для сгенерированных результатов, которые можно использовать для оценки прогресса обучения
- В следующих главах, посвященных тонкой настройке больших языковых моделей, мы также рассмотрим дополнительные способы оценки качества модели